# TP — Avion à décollage vertical (PVTOL)
**Dynamique des zéros instable, sortie plate et extension dynamique**

Master 3EA, parcours ASI — Commande non linéaire

Ce notebook accompagne l'énoncé `TP_PVTOL.pdf`. Les cellules marquées `# À COMPLÉTER` sont à écrire ; les réponses rédigées vont dans les cellules *Votre réponse*. Le notebook complété est votre rapport.

$$\ddot x = -\sin\theta\,u_1 + \varepsilon\cos\theta\,u_2,\qquad \ddot z = \cos\theta\,u_1 + \varepsilon\sin\theta\,u_2 - 1,\qquad \ddot\theta = u_2 .$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (11, 3.4)

## Modèle, trajectoire de référence et outils (fournis)

- État $X = (x, \dot x, z, \dot z, \theta, \dot\theta)$, commandes $u = (u_1, u_2)$ (poussée, couple).
- `reference(t)` donne la trajectoire désirée du centre de masse et ses dérivées jusqu'à l'ordre 4 : de $(0,0)$ à $(5,5)$ en `T_traj` secondes, avec des dérivées nulles aux deux bouts.
- `pvtol_dynamics(X, u, eps_reel, sature)` : avec `sature=True`, la poussée appliquée est $\max(u_1, 0)$.

In [ ]:
eps = 0.1                       # couplage couple -> force latérale
CIBLE = np.array([5.0, 5.0])    # point à rejoindre (x, z)

def reference(t, T_traj=8.0):
    """Renvoie un tableau 5 x 2 : y_d et ses dérivées d'ordre 1 à 4, pour (x, z) ; r[k] = dérivée d'ordre k.
    Polynôme de degré 9 : toutes les dérivées jusqu'à l'ordre 4 sont nulles en 0 et en T_traj."""
    tau = np.clip(t / T_traj, 0.0, 1.0)
    c = np.array([0, 0, 0, 0, 0, 126.0, -420.0, 540.0, -315.0, 70.0])
    p = np.polynomial.Polynomial(c)
    d = np.array([p.deriv(k)(tau) / T_traj**k if k else p(tau) for k in range(5)])
    return np.outer(d, CIBLE)

def pvtol_dynamics(X, u, eps_reel=eps, sature=False):
    """Équations du mouvement. u = (u1, u2). Si sature, la poussée ne peut pas être négative."""
    x, xd, z, zd, th, thd = X
    u1, u2 = u
    if sature:
        u1 = max(0.0, u1)
    s, c = np.sin(th), np.cos(th)
    return np.array([xd, -s * u1 + eps_reel * c * u2,
                     zd, c * u1 + eps_reel * s * u2 - 1.0,
                     thd, u2])

def centre_oscillation(X, e=eps):
    """Position du centre d'oscillation (CO), à la distance e au-dessus du centre de masse."""
    return np.array([X[0] - e * np.sin(X[4]), X[2] + e * np.cos(X[4])])

def trace(t, X, U, titre, T_traj=8.0, avec_CO=False):
    """Trajectoire dans le plan, angle et commandes."""
    ref = np.array([reference(tt, T_traj)[0] for tt in t])
    fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
    ax[0].plot(ref[:, 0], ref[:, 1], 'k:', label='référence')
    ax[0].plot(X[0], X[2], label='centre de masse')
    if avec_CO:
        CO = centre_oscillation(X)
        ax[0].plot(CO[0], CO[1], '--', label="centre d'oscillation")
    ax[0].set(xlabel='x', ylabel='z', title=titre); ax[0].legend(fontsize=7)
    ax[1].plot(t, X[0] - ref[:, 0], label='x - x_d'); ax[1].plot(t, X[2] - ref[:, 1], label='z - z_d')
    ax[1].set(xlabel='t (s)', title='erreur du centre de masse'); ax[1].legend(fontsize=7)
    ax[2].plot(t, np.rad2deg(X[4])); ax[2].set(xlabel='t (s)', ylabel=r'$\theta$ (°)')
    ax[3].plot(t, U[0], label='u1 (poussée)'); ax[3].plot(t, U[1], label='u2 (couple)')
    ax[3].set(xlabel='t (s)', title='commandes'); ax[3].legend(fontsize=7)
    for a in ax:
        a.grid(True)
    plt.tight_layout(); plt.show()

X0 = np.zeros(6)                # au sol, au repos

## Animation (fournie)
La fonction `anime` transforme une simulation en petit film. Elle sert à *voir* ce que fait la loi de commande ; elle n'est pas à modifier, mais vous pouvez l'appeler sur n'importe laquelle de vos simulations (utile pour la soutenance).

In [ ]:
from matplotlib import animation
from matplotlib import patches as mpatches
from IPython.display import HTML

def _film(fig, dessine, n_images, ips=20):
    """Fabrique le film : vidéo si ffmpeg est disponible (c'est le cas sur Colab), lecteur intégré sinon."""
    anim = animation.FuncAnimation(fig, dessine, frames=n_images, interval=1000 / ips, blit=False)
    plt.close(fig)
    try:
        return HTML(anim.to_html5_video())
    except Exception:
        return HTML(anim.to_jshtml())

def _instants(t, n_images):
    """Indices de n_images instants régulièrement espacés (le solveur, lui, a un pas variable)."""
    return np.clip(np.searchsorted(t, np.linspace(t[0], t[-1], n_images)), 0, len(t) - 1)

def anime(t, X, U, titre='', T_traj=8.0, avec_CO=False, n_images=100):
    """Film du vol (fourni). t, X, U : sorties de simule_S1 ou simule_S2 (X : les 6 premiers états).
    Flammes orange : poussée positive ; flammes bleues au-dessus : les moteurs « aspirent » (poussée négative)."""
    k = _instants(t, n_images)
    ref = np.array([reference(tt, T_traj)[0] for tt in t])
    lo, hi = min(0.0, CIBLE.min()) - 2.0, max(0.0, CIBLE.max()) + 2.0
    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel='x', ylabel='z', title=titre); ax.set_aspect('equal')
    ax.axhspan(lo, 0, color='0.88', zorder=0); ax.axhline(0, color='0.4', lw=1)
    ax.plot(ref[:, 0], ref[:, 1], 'k:', lw=1, label='référence')
    ax.plot(*CIBLE, marker='*', ms=16, color='gold', mec='k', zorder=3)
    trace_cm, = ax.plot([], [], color='C0', lw=1.5, label='centre de masse')
    trace_co, = ax.plot([], [], '--', color='C2', lw=1, label="centre d'oscillation")
    aile = mpatches.Polygon(np.zeros((4, 2)), closed=True, fc='0.3', ec='k', zorder=5)
    nez = mpatches.Polygon(np.zeros((3, 2)), closed=True, fc='C3', ec='k', zorder=6)
    flammes = [mpatches.Polygon(np.zeros((3, 2)), closed=True, zorder=4) for _ in range(2)]
    for p in (aile, nez, *flammes):
        ax.add_patch(p)
    point_co, = ax.plot([], [], 'o', color='C2', ms=5, zorder=7)
    texte = ax.text(0.03, 0.97, '', transform=ax.transAxes, va='top', family='monospace', fontsize=9)
    bulle = ax.text(0.5, 0.06, '', transform=ax.transAxes, ha='center', color='C3', fontsize=12, weight='bold')
    ax.legend(loc='lower right', fontsize=7)
    aile_b = np.array([[-0.75, -0.07], [0.75, -0.07], [0.75, 0.07], [-0.75, 0.07]])
    nez_b = np.array([[-0.16, 0.07], [0.16, 0.07], [0.0, 0.5]])

    def dessine(n):
        i = k[n]
        x, z, th = X[0, i], X[2, i], X[4, i]
        u1, u2 = U[0, i], U[1, i]
        R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
        pose = lambda pts: pts @ R.T + [x, z]
        aile.set_xy(pose(aile_b)); nez.set_xy(pose(nez_b))
        for fl, cote in zip(flammes, (-1, 1)):                    # un moteur sous chaque aile
            L = np.clip(0.45 * (u1 + cote * 0.12 * u2), -1.3, 1.3)
            base = np.array([[cote * 0.5 - 0.12, 0.0], [cote * 0.5 + 0.12, 0.0], [cote * 0.5, -L]])
            base[:2, 1] = -0.07 if L >= 0 else 0.07
            fl.set_xy(pose(base)); fl.set_facecolor('orange' if L >= 0 else 'deepskyblue'); fl.set_edgecolor('none')
        trace_cm.set_data(X[0, :i + 1], X[2, :i + 1])
        if avec_CO:
            co = centre_oscillation(X[:, :i + 1])
            trace_co.set_data(co[0], co[1]); point_co.set_data([co[0][-1]], [co[1][-1]])
        texte.set_text('t = %4.1f s\ntheta = %6.0f°\nu1 = %5.2f' % (t[i], np.rad2deg(th), u1))
        tours = abs(th) / (2 * np.pi)
        bulle.set_color('C3')
        if z < -0.05:
            bulle.set_text('sous le sol...')
        elif u1 < 0:
            bulle.set_text('les moteurs aspirent !')
        elif tours >= 1:
            bulle.set_text('%d tour%s sur lui-même' % (tours, 's' if tours >= 2 else ''))
        elif np.cos(th) < 0:
            bulle.set_text("sur le dos !")
        elif t[i] > T_traj and np.hypot(x - CIBLE[0], z - CIBLE[1]) < 0.02 and abs(th) < 0.02:
            bulle.set_text('arrivé, bien droit'); bulle.set_color('C2')
        else:
            bulle.set_text('')
    return _film(fig, dessine, n_images)

# Séance 1 — Linéarisation entrée-sortie sur le centre de masse

## Travail préparatoire (questions 1 à 3 de l'énoncé)

Ces questions se traitent sur papier, **avant** de programmer. Rédigez ensuite vos réponses ici : elles font partie du rapport. Les formules s'écrivent en LaTeX entre `$...$` (double-cliquez sur une cellule de l'énoncé pour voir des exemples). Donnez les étapes du calcul, pas seulement le résultat.

### Q1. Degré relatif

*Votre réponse :*

### Q2. Singularité statique

*Votre réponse :*

### Q3. Dynamique des zéros

*Votre réponse :*

## Travail pratique

### Q4. Prise en main
Lisez `pvtol_dynamics`. Vérifiez que le vol stationnaire ($\theta = 0$, $u = (1, 0)$) est un équilibre, puis simulez 3 s en boucle ouverte avec $u = (1, 0)$ depuis $\theta(0) = 2^\circ$.

In [ ]:
def simule_bo(theta0_deg, T=3.0):
    X_init = np.array([0, 0, 0, 0, np.deg2rad(theta0_deg), 0.0])
    sol = solve_ivp(lambda t, X: pvtol_dynamics(X, (1.0, 0.0)), (0, T), X_init, max_step=0.01)
    return sol.t, sol.y

print('f(X = 0, u = (1, 0)) =', pvtol_dynamics(np.zeros(6), (1.0, 0.0)))
t, X = simule_bo(2.0)
print('après 3 s : x = %.3f, z = %.3f, theta = %.1f°' % (X[0, -1], X[2, -1], np.rad2deg(X[4, -1])))

*Votre réponse (l'équilibre est-il stable ? que fait l'avion ?) :*

### Q5. Loi linéarisante sur le centre de masse
Avec $y = (x, z)$ : $\ddot y = \mathcal A(X) + \mathcal B(X)\,u$. Implémentez $u = \mathcal B^{-1}(v - \mathcal A)$ avec
$v = \ddot y_d + K_D(\dot y_d - \dot y) + K_P(y_d - y)$, $K_P = 4$, $K_D = 4$ (pôles doubles en $-2$).

In [ ]:
KP, KD = 4.0, 4.0

def controller_S1(t, X, T_traj=8.0, eps_modele=eps):
    """Linéarisation entrée-sortie, sortie = centre de masse. Renvoie (u1, u2)."""
    x, xd, z, zd, th, thd = X
    r = reference(t, T_traj)
    s, c = np.sin(th), np.cos(th)
    # À COMPLÉTER : A(X), B(X), commande virtuelle v, puis u = B^{-1} (v - A)
    return np.array([1.0, 0.0])

def simule_S1(T_sim=12.0, T_traj=8.0, sature=False, loi=None):
    loi = controller_S1 if loi is None else loi
    f = lambda t, X: pvtol_dynamics(X, loi(t, X, T_traj), sature=sature)
    sol = solve_ivp(f, (0, T_sim), X0, max_step=0.005, rtol=1e-8, atol=1e-10)
    U = np.array([loi(tt, sol.y[:, j], T_traj) for j, tt in enumerate(sol.t)]).T
    if sature:
        U[0] = np.maximum(U[0], 0.0)
    return sol.t, sol.y, U

### Q6. Simulation : le centre de masse suit, et l'angle ?

In [ ]:
t, X, U = simule_S1()
trace(t, X, U, 'S1 : sortie = centre de masse')
print('erreur max du centre de masse : %.1e' % np.abs(np.r_[X[0] - [reference(tt)[0, 0] for tt in t], X[2] - [reference(tt)[0, 1] for tt in t]]).max())
print('theta : max |theta| = %.0f°, valeur finale %.0f° ; poussée minimale %.2f' % (np.rad2deg(np.abs(X[4]).max()), np.rad2deg(X[4, -1]), U[0].min()))

In [ ]:
anime(t, X, U, titre='S1 : on pilote le centre de masse')

*Votre réponse (que fait $\theta$ ? la poussée reste-t-elle positive ? reliez à la dynamique des zéros $\ddot\theta = \sin\theta/\varepsilon$) :*

### Q7. Poussée positive seulement
Même simulation avec `sature=True` : la poussée appliquée est $\max(u_1, 0)$.

In [ ]:
t, X, U = simule_S1(sature=True)
trace(t, X, U, 'S1, poussée >= 0')
print('altitude finale : %.1f' % X[2, -1])

In [ ]:
anime(t, X, U, titre='S1, poussée positive seulement')

*Votre réponse :*

### Q8. Peut-on s'en sortir par le réglage ?
Essayez des gains plus doux ($K_P = 1$, $K_D = 2$) et une trajectoire plus lente (`T_traj = 15`). Mesurez l'instant où $|\theta|$ dépasse $90^\circ$.

In [ ]:
def instant_bascule(t, X):
    i = np.where(np.abs(X[4]) > np.pi / 2)[0]
    return t[i[0]] if len(i) else np.inf

# À COMPLÉTER : refaire la simulation pour plusieurs (KP, KD, T_traj) et afficher instant_bascule(t, X)
# (KP et KD sont des variables globales lues par controller_S1 ; remettez KP = KD = 4 à la fin)

*Votre réponse (le réglage change-t-il la conclusion ? pourquoi ?) :*

# Séance 2 — Sortie plate et extension dynamique

On pilote le centre d'oscillation $Y = (x - \varepsilon\sin\theta,\ z + \varepsilon\cos\theta)$. Avec $u_1 = \xi_1 + \varepsilon\dot\theta^2$ et deux intégrateurs $\dot\xi_1 = \xi_2$, $\dot\xi_2 = v_1$ :

$$Y^{(4)} = \mathcal B_{dyn}\begin{pmatrix}v_1\\u_2\end{pmatrix} + \mathcal A_{exact}.$$

**Référence.** Au repos ($\theta = 0$), le centre d'oscillation est à $\varepsilon$ **au-dessus** du centre de masse. Pour que le centre de masse aille de $(0,0)$ à $(5,5)$, la référence du centre d'oscillation est donc $y_d(t) + (0, \varepsilon)$.

## Travail préparatoire (questions 9 à 12 de l'énoncé)

Ces questions se traitent sur papier, **avant** de programmer. Rédigez ensuite vos réponses ici : elles font partie du rapport. Les formules s'écrivent en LaTeX entre `$...$` (double-cliquez sur une cellule de l'énoncé pour voir des exemples). Donnez les étapes du calcul, pas seulement le résultat.

### Q9. Accélération de la sortie

*Votre réponse :*

### Q10. Simplification par changement de variable

*Votre réponse :*

### Q11. Extension dynamique

*Votre réponse :*

### Q12. Singularité et interprétation physique

*Votre réponse :*

## Travail pratique

### Q13. La loi
État étendu $X_{aug} = (X, \xi_1, \xi_2)$. Gains de $(s+\lambda)^4$ avec $\lambda = 3$. Le paramètre `eps_modele` est la valeur de $\varepsilon$ que **croit** le correcteur (utile à la question 15).

In [ ]:
lam = 3.0
K0, K1, K2, K3 = lam**4, 4 * lam**3, 6 * lam**2, 4 * lam      # (s + lam)^4

def controller_S2(t, Xa, T_traj=8.0, eps_modele=eps, decalage=True):
    """Extension dynamique sur le centre d'oscillation. Renvoie (u1, u2, v1)."""
    x, xd, z, zd, th, thd, xi1, xi2 = Xa
    s, c = np.sin(th), np.cos(th)
    e = eps_modele
    r = reference(t, T_traj).copy()
    if decalage:
        r[0, 1] += e                                  # le CO vise eps au-dessus du point voulu
    # À COMPLÉTER :
    #  1. Y0, Y1, Y2, Y3 : la sortie (centre d'oscillation) et ses trois premières dérivées,
    #     en fonction de l'état étendu (formules de la préparation) ;
    #  2. W = r[4] - K3 (Y3 - r[3]) - ... : commande virtuelle, avec le terme d'anticipation r[4] ;
    #  3. B_dyn, A_exact, puis (v1, u2) = solution de B_dyn (v1, u2) = W - A_exact ;
    #  4. u1 = xi1 + e * thd**2.
    return 1.0, 0.0, 0.0

def simule_S2(T_sim=12.0, T_traj=8.0, eps_modele=eps, xi1_0=1.0, decalage=True, sature=True):
    def f(t, Xa):
        u1, u2, v1 = controller_S2(t, Xa, T_traj, eps_modele, decalage)
        return np.r_[pvtol_dynamics(Xa[:6], (u1, u2), sature=sature), Xa[7], v1]
    stop = lambda t, Xa: 50.0 - np.abs(Xa[:6]).max()          # arrêt si la simulation diverge
    stop.terminal = True
    sol = solve_ivp(f, (0, T_sim), np.r_[X0, xi1_0, 0.0], max_step=0.005, rtol=1e-8, atol=1e-10, events=stop)
    U = np.array([controller_S2(tt, sol.y[:, j], T_traj, eps_modele, decalage)[:2] for j, tt in enumerate(sol.t)]).T
    return sol.t, sol.y, U

### Q14. Simulation et analyse

In [ ]:
t, Xa, U = simule_S2()
trace(t, Xa[:6], U, 'S2 : sortie = centre d\'oscillation', avec_CO=True)
ref = np.array([reference(tt)[0] for tt in t])
err = np.hypot(Xa[0] - ref[:, 0], Xa[2] - ref[:, 1])
print('centre de masse : écart max à la référence %.3f (pendant le mouvement), écart final %.1e' % (err.max(), err[-1]))
print('theta : max %.1f°, final %.1e° ; poussée u1 entre %.2f et %.2f' % (np.rad2deg(np.abs(Xa[4]).max()), np.rad2deg(Xa[4, -1]), U[0].min(), U[0].max()))

t2, Xb, U2 = simule_S2(decalage=False)
print('sans décalage de la référence : position finale du centre de masse = (%.3f, %.3f)' % (Xb[0, -1], Xb[2, -1]))

In [ ]:
anime(t, Xa[:6], U, titre="S2 : on pilote le centre d'oscillation", avec_CO=True)

*Votre réponse : (a) le centre d'oscillation suit-il la référence ? et le centre de masse ? (b) que devient $\theta$ ? (c) la poussée reste-t-elle positive ? (d) que se passe-t-il sans le décalage de $\varepsilon$ ? (e) essayez `xi1_0 = 0` et expliquez.*

### Q15. Robustesse : $\varepsilon$ mal connu
Le correcteur croit que le couplage vaut `eps_modele`, alors que l'avion a $\varepsilon = 0{,}1$. Le point qu'il pilote n'est donc plus le vrai centre d'oscillation.

In [ ]:
def regle_gains(l):
    "Place les quatre pôles de l'erreur en -l (modifie les gains globaux lus par controller_S2)."
    global K0, K1, K2, K3
    K0, K1, K2, K3 = l**4, 4 * l**3, 6 * l**2, 4 * l

def bilan(t, Xa):
    "Oscillation résiduelle de theta après t = 30 s (en degrés) et écart final du centre de masse à la cible."
    fin = t > 30.0
    osc = np.rad2deg(np.abs(Xa[4][fin]).max()) if fin.any() else np.inf
    return osc, np.hypot(Xa[0, -1] - CIBLE[0], Xa[2, -1] - CIBLE[1])

# À COMPLÉTER : pour lambda = 3, simuler (T_sim = 40, sature=False) avec eps_modele dans
# (-0.05, 0, 0.05, 0.1, 0.15, 0.3) et afficher bilan(t, Xa) et la poussée minimale ;
# puis refaire eps_modele = 0 avec lambda = 2 et lambda = 5 (regle_gains), et eps_modele = 0.05 avec lambda = 5.
# Remettre regle_gains(3.0) à la fin.

*Votre réponse : vaut-il mieux surestimer ou sous-estimer $\varepsilon$ ? Quel rôle joue $\lambda$ ? Indication : pour la sortie $(x - \hat\varepsilon\sin\theta,\ z + \hat\varepsilon\cos\theta)$, refaites le calcul de la dynamique des zéros de la séance 1.*

### Question ouverte (une au choix, obligatoire)
Choisissez une des trois questions de l'énoncé (manœuvre plus rapide, suivi exact du centre de masse, comparaison avec un retour d'état linéaire) et traitez-la ici. Elle n'est pas guidée et doit apparaître dans la soutenance.

In [ ]:
# Votre travail